# Lab 13: Information Retrieval for RAG

## 1. What is Information Retrieval?

**Information retrieval (IR)** means finding relevant information from a collection of documents based on a user query.

**Example:** A student asks, *"How do transformers process words?"* The retrieval system searches course notes and returns the most relevant passages.

**Simple process:**

`Documents → Numerical representation → Compare with query → Rank → Top results`

## 2. Create a small document collection

Imagine these are short notes taken from an NLP course. Each string represents one document.

In [4]:
documents = [
    "Transformers use self-attention to understand relationships between words.",
    "TF-IDF represents documents using word importance scores.",
    "Word2Vec learns dense vector representations of words.",
    "Named entity recognition identifies names, places, and organizations in text.",
    "Information retrieval finds relevant documents for a user query.",
    "Cosine similarity measures how similar two text vectors are."]

for i, document in enumerate(documents):
    print(f"Document {i}: {document}")

Document 0: Transformers use self-attention to understand relationships between words.
Document 1: TF-IDF represents documents using word importance scores.
Document 2: Word2Vec learns dense vector representations of words.
Document 3: Named entity recognition identifies names, places, and organizations in text.
Document 4: Information retrieval finds relevant documents for a user query.
Document 5: Cosine similarity measures how similar two text vectors are.


## 3. Convert documents into TF-IDF vectors (sparse retrieval)

A computer cannot compare raw text directly using vector similarity. **TF-IDF** converts each document into a numerical vector.

- **TF** measures how frequently a word occurs in a document.
- **IDF** gives more importance to words that appear in fewer documents.

TF-IDF is a form of **sparse retrieval**: most word positions in the vectors are zero.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(documents)

print("TF-IDF matrix shape:", document_vectors.shape)
print("First document vector:", document_vectors[0].toarray().round(2))

TF-IDF matrix shape: (6, 39)
First document vector: [[0.39 0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
  0.   0.   0.   0.   0.39 0.   0.   0.   0.   0.   0.39 0.   0.   0.
  0.   0.39 0.39 0.39 0.   0.   0.   0.   0.   0.   0.32]]


**Check:** The matrix has one row per document and one column per vocabulary word. A zero means that word has no TF-IDF weight in that document.

## 4. Search using a query

We use the **same TF-IDF vectorizer** to represent the user's query. Then we calculate **cosine similarity** between the query and every document.

- Higher similarity → more relevant according to TF-IDF.
- A similarity of 0 means the vectors do not share weighted terms.

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

query = "How do transformers understand words?"
query_vector = vectorizer.transform([query])
scores = cosine_similarity(query_vector, document_vectors)[0]

for i, score in enumerate(scores):
    print(f"Document {i}: {score:.3f}")

Document 0: 0.633
Document 1: 0.000
Document 2: 0.173
Document 3: 0.000
Document 4: 0.000
Document 5: 0.000


## 5. Retrieve the top-k documents

Rather than reading every document, the system returns the **k** documents with the highest similarity scores.

In [7]:
k = 3
ranked_ids = scores.argsort()[::-1][:k]

for rank, i in enumerate(ranked_ids, start=1):
    print(f"Rank {rank} | Score: {scores[i]:.3f}")
    print(documents[i])
    print()

Rank 1 | Score: 0.633
Transformers use self-attention to understand relationships between words.

Rank 2 | Score: 0.173
Word2Vec learns dense vector representations of words.

Rank 3 | Score: 0.000
Information retrieval finds relevant documents for a user query.



## 6. Student exercises

**Exercise 1 — Change the query**  
Use `search()` to find notes about **named entity recognition**. Which document is ranked first?

**Exercise 2 — Change k**  
Search for `"document representation"` using `k=1` and then `k=3`. What changes?


In [8]:
# The exercises ask us to use search(), so we first define it
def search(query, k=3):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    top_ids = scores.argsort()[::-1][:k]

    for rank, i in enumerate(top_ids, start=1):
        print(f"Rank {rank} | Score: {scores[i]:.3f} | {documents[i]}")

# Exercise 1: find notes about named entity recognition
search("named entity recognition", k=3)

Rank 1 | Score: 0.625 | Named entity recognition identifies names, places, and organizations in text.
Rank 2 | Score: 0.000 | Cosine similarity measures how similar two text vectors are.
Rank 3 | Score: 0.000 | Information retrieval finds relevant documents for a user query.


**Answer (Exercise 1):**
**Document 3** ("Named entity recognition identifies names, places, and organizations in text.") is ranked first with a score of about **0.625**. It is the only document that contains the query words (*named*, *entity*, *recognition*), so it is the only one with a score above zero. The other results have a score of 0.000, so they are not really relevant.

In [9]:
# Exercise 2: the query "document representation" with k=1 and k=3
print("k = 1")
search("document representation", k=1)

print("\nk = 3")
search("document representation", k=3)

k = 1
Rank 1 | Score: 0.000 | Cosine similarity measures how similar two text vectors are.

k = 3
Rank 1 | Score: 0.000 | Cosine similarity measures how similar two text vectors are.
Rank 2 | Score: 0.000 | Information retrieval finds relevant documents for a user query.
Rank 3 | Score: 0.000 | Named entity recognition identifies names, places, and organizations in text.


**Answer (Exercise 2):**
Changing `k` only changes **how many** documents are returned: one result for `k=1` and three results for `k=3`. But look at the scores: they are all **0.000**, so none of these documents is really a match, and the document shown first is just an arbitrary one among the documents with the same score.

The reason is that TF-IDF matches **exact words**. The documents contain "document**s**", "representation**s**" and "represent**s**", but not the exact words "document" and "representation", and TF-IDF does not know that these words have the same root. A larger `k` does not make the results better, it only adds more (equally irrelevant) documents. This is a limitation of sparse retrieval, and one reason why dense retrieval (embeddings) is useful: it can match by **meaning**, not only by exact words.

## Summary

You built a small **information retrieval** system using TF-IDF and cosine similarity. It converts documents and queries into vectors, ranks documents by relevance, and returns the top results.

**Connection to RAG:** These retrieved documents can later be passed to a language model as supporting context for answer generation.

---
# Part 2 starts here

Run the *Setup* cell below first: it recreates the document collection and TF-IDF vectors.

# Lab 13 — Part 2: Search Functions and Retrieval Approaches

**Learning goals:** Reuse the retrieval pipeline in a function, explore different queries, and compare sparse and dense retrieval.

## Setup — Run this cell first

The cell below recreates the small document collection and TF-IDF vectors used in Part 1.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Transformers use self-attention to understand relationships between words.",
    "TF-IDF represents documents using word importance scores.",
    "Word2Vec learns dense vector representations of words.",
    "Named entity recognition identifies names, places, and organizations in text.",
    "Information retrieval finds relevant documents for a user query.",
    "Cosine similarity measures how similar two text vectors are."
]

vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(documents)

## 1. Make a simple search function

Now combine the steps so you can search the document collection using any query.

In [11]:
def search(query, k=2):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    top_ids = scores.argsort()[::-1][:k]

    for i in top_ids:
        print(f"Score: {scores[i]:.3f} | {documents[i]}")

search("How can we measure similarity between text vectors?")

Score: 0.686 | Cosine similarity measures how similar two text vectors are.
Score: 0.148 | Named entity recognition identifies names, places, and organizations in text.


## 2. Sparse vs. dense retrieval (conceptual comparison)

| Sparse retrieval | Dense retrieval |
|---|---|
| Represents text using word-based features, e.g., TF-IDF | Represents text using learned embeddings |
| Works well when important query words appear in documents | Can identify similar meanings even when different words are used |
| Simple and lightweight | Usually requires a pretrained embedding model |

**Important limitation:** TF-IDF mainly matches words, not meanings. A query about *"automobiles"* may not retrieve a document about *"cars"* unless the words also overlap in other ways.

In modern RAG, retrieval can be sparse, dense, or a combination of both.

## 3. Does a high-ranked result always mean a good match?

The search function returns the **highest-scoring** documents, even when all the scores are zero. A high *rank* does not always mean a high *similarity score*.

For example, our collection is about NLP. What happens when we search for an unrelated topic like `"volcano eruption"`?

In [12]:
search("volcano eruption", k=2)

# Compare it with a relevant query
print("--- Relevant query ---")
search("cosine similarity", k=2)

Score: 0.000 | Cosine similarity measures how similar two text vectors are.
Score: 0.000 | Information retrieval finds relevant documents for a user query.
--- Relevant query ---
Score: 0.594 | Cosine similarity measures how similar two text vectors are.
Score: 0.000 | Information retrieval finds relevant documents for a user query.


## 4. Improve search: skip results with a score of zero

We can improve the function by printing only documents with a **positive similarity score**.

In [13]:
def search_filtered(query, k=2):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    top_ids = scores.argsort()[::-1][:k]

    found = False
    for i in top_ids:
        if scores[i] > 0:
            print(f"Score: {scores[i]:.3f} | {documents[i]}")
            found = True
    if not found:
        print("No matching documents found.")

search_filtered("volcano eruption")
print("---")
search_filtered("cosine similarity")

No matching documents found.
---
Score: 0.594 | Cosine similarity measures how similar two text vectors are.


## 5. Practice questions

1. Use `search()` to find a note about **named entity recognition**. Which document ranks first?
2. Add `"BERT is a transformer model that uses bidirectional context."` to `documents`, rebuild the TF-IDF vectors using `vectorizer.fit_transform(documents)`, and search for `"transformer models"`. Why was rebuilding necessary?

In [14]:
# Practice: run your own queries here
search("named entity recognition", k=2)

Score: 0.625 | Named entity recognition identifies names, places, and organizations in text.
Score: 0.000 | Cosine similarity measures how similar two text vectors are.


**Answer (Practice 1):**
The query `"named entity recognition"` ranks **Document 3** ("Named entity recognition identifies names, places, and organizations in text.") first, with a score of about **0.625**. It contains all the words of the query. The second result has a score of 0.000, so it is not a real match.

In [15]:
# Practice 2: add the BERT document and rebuild the TF-IDF vectors

# Before rebuilding: the matrix does not know the new document
documents.append("BERT is a transformer model that uses bidirectional context.")
print("Documents in the list      :", len(documents))
print("Rows in the TF-IDF matrix  :", document_vectors.shape[0])
print("'bert' in the vocabulary?  :", "bert" in vectorizer.vocabulary_)

# Rebuild the vocabulary, the IDF values and the vectors for ALL the documents
document_vectors = vectorizer.fit_transform(documents)

print("\nAfter rebuilding")
print("Rows in the TF-IDF matrix  :", document_vectors.shape[0])
print("'bert' in the vocabulary?  :", "bert" in vectorizer.vocabulary_)

print("\nSearch results for 'transformer models':")
search("transformer models", k=3)

Documents in the list      : 7
Rows in the TF-IDF matrix  : 6
'bert' in the vocabulary?  : False

After rebuilding
Rows in the TF-IDF matrix  : 7
'bert' in the vocabulary?  : True

Search results for 'transformer models':
Score: 0.408 | BERT is a transformer model that uses bidirectional context.
Score: 0.000 | Cosine similarity measures how similar two text vectors are.
Score: 0.000 | Information retrieval finds relevant documents for a user query.


**Answer (Practice 2):**
Rebuilding was necessary because the `TfidfVectorizer` and `document_vectors` were created from the **original 6 documents only**. After appending the new sentence to `documents`, the matrix still had 6 rows and the vocabulary did not contain words like `bert` or `bidirectional`, so the new document had no vector and could never be retrieved. `vectorizer.fit_transform(documents)` learns the vocabulary and the IDF values again from the whole collection (now 7 documents) and creates a row for every document.

After rebuilding, the BERT document is ranked first with a score of about **0.408**, because it contains the word "transformer". The original note about "Transformers" scores 0 for this query, because TF-IDF treats "transformer" and "transformers" as different words (and the query word "models" does not match "model" either). Again, this shows that sparse retrieval depends on exact word matches.